# SmolVLA + LIBERO Quickstart (Kaggle)

This notebook is a Kaggle-friendly draft for running a compact `SmolVLA + LIBERO` experiment.

Use this version when you want:
- a cleaner long-running GPU session than free Colab
- a small benchmark run on `libero_10`
- exported logs and figures


In [ ]:
!nvidia-smi
%cd /kaggle/working
!git clone https://github.com/huggingface/lerobot.git
%cd /kaggle/working/lerobot
!pip install -q -U pip
!pip install -q -e ".[smolvla]"
!pip install -q -e ".[libero]"


In [ ]:
import os

os.environ["MUJOCO_GL"] = "egl"
HF_USER = "your_hf_username"
POLICY_REPO_ID = f"{HF_USER}/libero-smolvla-kaggle"

print("Use Kaggle secrets or `huggingface-cli login` if you need to push a checkpoint.")


## Fine-tune run

This follows the current official LIBERO training interface from LeRobot, but starts with a smaller training budget.

In [ ]:
%%bash
cd /kaggle/working/lerobot
export MUJOCO_GL=egl
HF_USER=your_hf_username
POLICY_REPO_ID=${HF_USER}/libero-smolvla-kaggle
lerobot-train \
  --policy.type=smolvla \
  --policy.repo_id=${POLICY_REPO_ID} \
  --policy.load_vlm_weights=true \
  --dataset.repo_id=HuggingFaceVLA/libero \
  --env.type=libero \
  --env.task=libero_10 \
  --output_dir=./outputs/libero_smolvla \
  --steps=20000 \
  --batch_size=4 \
  --eval.batch_size=1 \
  --eval.n_episodes=1 \
  --eval_freq=1000


## Evaluate a checkpoint

In [ ]:
%%bash
cd /kaggle/working/lerobot
export MUJOCO_GL=egl
POLICY_PATH=your_hf_username/libero-smolvla-kaggle
lerobot-eval \
  --output_dir=./eval_logs/libero_eval \
  --env.type=libero \
  --env.task=libero_spatial,libero_object,libero_goal,libero_10 \
  --eval.batch_size=1 \
  --eval.n_episodes=2 \
  --policy.path=${POLICY_PATH} \
  --policy.n_action_steps=10 \
  --env.max_parallel_tasks=1


## Suggested Kaggle deliverables

- export checkpoints or metrics
- copy aggregated results back into this repo
- generate plots with `scripts/analyze_results.py`
- add a MuJoCo desktop sorting showcase video for the project homepage
